# Dashboard Report Generator

This notebook demonstrates how to generate dashboard reports in **Excel** and **CSV** formats.

## Features:
* Create reports from Spark DataFrames or pandas DataFrames
* Export to CSV (simple, fast)
* Export to Excel with formatting (styled reports)
* Support for multiple sheets in Excel
* Automatic date formatting and column widths

In [0]:
dbutils.widgets.text("env", "dev", "env")
env = dbutils.widgets.get("env")
current_date = datetime.now().strftime('%Y%m%d')


In [0]:
# Install openpyxl for Excel support with styling
%pip install openpyxl

In [0]:
import pandas as pd

# Load data from Finnhub tables
print("Loading data from Finnhub tables...")

# Read current stocks report
stocks_report_df = spark.table(f"cadp_m8972_finnhub_dp_{env}.finnhub_stocks_report")
df = stocks_report_df.toPandas()

# Read stocks report history
stocks_history_df = spark.table(f"cadp_m8972_finnhub_dp_{env}.finnhub_stocks_report_history")
history_df = stocks_history_df.toPandas()

print(f"\nCurrent Stocks Report: {len(df):,} rows")
print(f"Stocks Report History: {len(history_df):,} rows")

# Display sample data
print("\n=== Current Stocks Report (First 10 rows) ===")
display(df.head(10))

print("\n=== Stocks Report History (First 10 rows) ===")
display(history_df.head(10))

# Create a summary
if not df.empty:
    print("\n=== Data Summary ===")
    print(f"Columns in current report: {list(df.columns)}")
    print(f"Columns in history: {list(history_df.columns)}")

In [0]:
csv_current_path = f'/Workspace/Users/girianiruddha4@gmail.com/Finnhub/notebooks/consumption_CADP/finnhub_stocks_report{current_date}.csv'
df.to_csv(csv_current_path, index=False)
print(f"✓ Current stocks report exported to: {csv_current_path}")

# Export stocks history report
csv_history_path = f'/Workspace/Users/girianiruddha4@gmail.com/Finnhub/notebooks/consumption_CADP/finnhub_stocks_report_history_{current_date}.csv'
history_df.to_csv(csv_history_path, index=False)
print(f"✓ Stocks history report exported to: {csv_history_path}")

In [0]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime

# Export to Excel with multiple sheets
excel_path = f'/Workspace/Users/girianiruddha4@gmail.com/Finnhub/notebooks/consumption_CADP/finnhub_dashboard_{current_date}.xlsx'

print("Creating Excel workbook...")

# Create Excel writer
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Write current stocks report
    df.to_excel(writer, sheet_name='Current Stocks', index=False)
    print(f"  → Sheet 'Current Stocks': {len(df):,} rows")
    
    # Write stocks history
    history_df.to_excel(writer, sheet_name='Stocks History', index=False)
    print(f"  → Sheet 'Stocks History': {len(history_df):,} rows")

print(f"\n✓ Excel workbook created: {excel_path}")

# Apply formatting to the Excel file
print("\nApplying formatting...")
wb = load_workbook(excel_path)

# Format each sheet
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    
    # Style header row
    header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF")
    
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')
    
    # Auto-adjust column widths
    for idx, column in enumerate(ws.columns, 1):
        max_length = 0
        column_letter = get_column_letter(idx)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        ws.column_dimensions[column_letter].width = adjusted_width
    
    # Add borders
    thin_border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )
    
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.border = thin_border

# Save formatted workbook
wb.save(excel_path)
print(f"✓ Formatting applied to Excel dashboard")
print(f"\n✓ Complete! Excel file ready at: {excel_path}")

In [0]:
# If you're working with Spark DataFrames, convert to pandas first

# Example: Read from a Delta table and export
# spark_df = spark.table("catalog.schema.table_name")

# Convert Spark DataFrame to pandas (for smaller datasets)
# pandas_df = spark_df.toPandas()

# For large datasets, use sampling or aggregation first
# sampled_df = spark_df.sample(fraction=0.1).toPandas()
# aggregated_df = spark_df.groupBy("column").agg({"metric": "sum"}).toPandas()

# Then export using the methods above
# pandas_df.to_csv('report.csv', index=False)
# pandas_df.to_excel('report.xlsx', index=False)

print("Tip: For large Spark DataFrames:")
print("  1. Aggregate or sample data first")
print("  2. Use .toPandas() to convert to pandas")
print("  3. Then export to CSV or Excel")
print("  4. For very large reports, consider partitioned CSV exports")